In [1]:
# ============================================================
# EMAIL/SMS SPAM DETECTION — single-cell pipeline
# ============================================================
import re, string, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 90)

# ---------- 1. LOAD DATA ----------
# Replace with the real Kaggle/UCI file if you have it:
# df = pd.read_csv("spam.csv", encoding="latin-1")[["v1","v2"]]
# df.columns = ["label","message"]
df = pd.read_csv("sms_spam.csv")[["label", "message"]].dropna().drop_duplicates().reset_index(drop=True)
print("Shape:", df.shape)

# ---------- 2. CLASS DISTRIBUTION ----------
counts = df["label"].value_counts()
pct = df["label"].value_counts(normalize=True) * 100
print("\nCounts:\n", counts.to_string())
print("\nPercentages:", {k: f"{v:.2f}%" for k, v in pct.items()})

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(counts.index, counts.values, color=["#4C72B0", "#DD8452"])
ax.set_title("Class Distribution: Spam vs Ham"); ax.set_ylabel("Number of messages")
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 3, str(int(b.get_height())), ha="center")
plt.tight_layout(); plt.show()

# ---------- 3. TEXT PREPROCESSING ----------
STOPWORDS = ENGLISH_STOP_WORDS  # swap for nltk.corpus.stopwords.words('english') if you prefer NLTK

def simple_stem(word):
    for suf in ("ational","ization","fulness","ousness","iveness","ing","edly","ies","ied","ed","es","ly","s"):
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            return word[: -len(suf)]
    return word

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = [t for t in text.split() if t not in STOPWORDS and len(t) > 1]
    return " ".join(simple_stem(t) for t in tokens)

df["clean_message"] = df["message"].apply(clean_text)
print("\nExample cleaning:")
print("BEFORE:", df["message"].iloc[0])
print("AFTER :", df["clean_message"].iloc[0])

# ---------- 4. TF-IDF FEATURES ----------
# TF-IDF weighs a word highly if it's frequent in a message but rare across
# the whole corpus (TF x IDF) — exactly the profile of discriminative words
# like "claim" or "winner", while common/stopword-like terms are down-weighted.
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2)
X = tfidf.fit_transform(df["clean_message"])
y = (df["label"] == "spam").astype(int)
print("\nTF-IDF matrix shape:", X.shape)

# ---------- 5. TRAIN/TEST SPLIT ----------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)
print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

# ---------- 6. TRAIN MODELS ----------
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Linear SVM": LinearSVC(class_weight="balanced", random_state=RANDOM_STATE),
}
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    predictions[name] = model.predict(X_test)

# ---------- 7. EVALUATION ----------
results = []
for name, preds in predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-score": f1_score(y_test, preds),
    })
results_df = pd.DataFrame(results).set_index("Model").round(4)
print("\n", results_df)

fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 4))
for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds)
    ax.imshow(cm, cmap="Blues")
    ax.set_title(name, fontsize=10)
    ax.set_xticks([0,1]); ax.set_xticklabels(["ham","spam"])
    ax.set_yticks([0,1]); ax.set_yticklabels(["ham","spam"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i,j], ha="center", va="center",
                     color="white" if cm[i,j] > cm.max()/2 else "black")
plt.tight_layout(); plt.show()

best = results_df["F1-score"].idxmax()
print(f"\nClassification report — {best}\n")
print(classification_report(y_test, predictions[best], target_names=["ham","spam"]))

# ---------- 8. DISCUSSION: why recall matters ----------

# ---------- 9. BONUS: word clouds (no external 'wordcloud' package needed) ----------
def top_words(label, n=40):
    text = " ".join(df.loc[df["label"] == label, "clean_message"])
    return Counter(text.split()).most_common(n)

def draw_pseudo_wordcloud(ax, word_freqs, title, cmap):
    rng = np.random.RandomState(RANDOM_STATE)
    max_c = word_freqs[0][1] if word_freqs else 1
    ax.set_xlim(0,10); ax.set_ylim(0,10); ax.axis("off"); ax.set_title(title)
    placed = []
    for word, c in word_freqs:
        size = 9 + 32*(c/max_c)
        for _ in range(30):
            x, y = rng.uniform(0.5,9.5), rng.uniform(0.5,9.5)
            if all((x-px)**2+(y-py)**2 > 0.55 for px,py in placed):
                placed.append((x,y)); break
        ax.text(x, y, word, fontsize=size, ha="center", va="center",
                 color=cmap(0.35+0.65*(c/max_c)), fontweight="bold")

spam_words, ham_words = top_words("spam"), top_words("ham")
fig, axes = plt.subplots(1, 2, figsize=(14,6))
draw_pseudo_wordcloud(axes[0], spam_words, "Most frequent words — SPAM", plt.cm.Reds)
draw_pseudo_wordcloud(axes[1], ham_words, "Most frequent words — HAM", plt.cm.Blues)
plt.tight_layout(); plt.show()
print("Top spam words:", spam_words[:10])
print("Top ham words :", ham_words[:10])

FileNotFoundError: [Errno 2] No such file or directory: 'sms_spam.csv'